# M16 · Dimensionality reduction & anomaly detection

Curriculum · Domain 3 · Unsupervised learning

**Compress high-dimensional behavior into a useful view, then spot points that do not belong.**

We will use PCA to map synthetic ads metrics, inspect explained variance, and flag anomalies with reconstruction error and IsolationForest.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

rng = np.random.default_rng(16)

## Build high-dimensional campaign metrics

Each row is a campaign-day. The base variables create correlated impressions, clicks, spend, conversion, and creative-quality signals.

In [ ]:
n_normal = 500
latent = rng.normal(size=(n_normal, 2))
noise = rng.normal(scale=0.25, size=(n_normal, 6))

mix = np.array([
    [1.0, 0.2, 0.8, 0.1, 0.5, -0.2],
    [0.1, 1.0, 0.2, 0.9, -0.4, 0.6],
])

X_normal = latent @ mix + noise
X_normal = StandardScaler().fit_transform(X_normal)

print("normal rows:", X_normal.shape[0])
print("features:", X_normal.shape[1])

## PCA in one formula

PCA centers $X$, computes $S = \frac{1}{n-1}X_c^\top X_c$, and projects onto eigenvectors with the largest eigenvalues. The first two components are the best two-dimensional linear view by variance.

In [ ]:
pca = PCA(n_components=2, random_state=16)
Z = pca.fit_transform(X_normal)
explained = pca.explained_variance_ratio_

print("explained variance:", np.round(explained, 3))
print("total kept:", round(float(explained.sum()), 3))

assert explained.sum() > 0.65

## Add anomalous campaign-days

We create days with unusual spend and click patterns. A good detector should rank many of these near the top.

In [ ]:
anomalies = np.array([
    [4.5, -3.5, 4.0, -3.0, 3.5, -2.5],
    [5.0, -4.0, 4.5, -3.5, 4.0, -3.0],
    [-4.0, 4.5, -3.5, 4.0, -3.0, 3.5],
    [-4.5, 5.0, -4.0, 4.5, -3.5, 4.0],
    [0.0, 0.0, 5.0, -5.0, 0.0, 0.0],
])

X_all = np.vstack([X_normal, anomalies])
is_anomaly = np.zeros(X_all.shape[0], dtype=bool)
is_anomaly[-len(anomalies):] = True

print("total rows:", X_all.shape[0])
print("injected anomalies:", int(is_anomaly.sum()))

## Reconstruction error

If normal campaign-days lie close to a two-dimensional linear subspace, projecting down and back up should reconstruct them well. Poor reconstruction is an anomaly signal: $\|x - \hat{x}\|_2^2$.

In [ ]:
Z_all = pca.transform(X_all)
X_hat = pca.inverse_transform(Z_all)
recon_error = np.sum((X_all - X_hat) ** 2, axis=1)
threshold = np.quantile(recon_error[:n_normal], 0.99)
flagged = recon_error > threshold

print("training p99 threshold:", round(float(threshold), 3))
print("flagged injected anomalies:", int(np.sum(flagged & is_anomaly)))

assert np.sum(flagged & is_anomaly) >= 3

## IsolationForest comparison

IsolationForest looks for points that are easy to isolate by random splits. It is nonlinear and often works well as a general-purpose tabular baseline.

In [ ]:
forest = IsolationForest(contamination=0.02, random_state=16)
forest.fit(X_normal)

forest_pred = forest.predict(X_all)
forest_flagged = forest_pred == -1

print("forest flagged total:", int(forest_flagged.sum()))
print("forest flagged injected:", int(np.sum(forest_flagged & is_anomaly)))

assert np.sum(forest_flagged & is_anomaly) >= 4

## Plot normal rows and injected outliers

We project the injected points into the PCA view. Points far from the cloud are obvious; points inside the cloud may still have high reconstruction error in discarded dimensions.

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
ax.scatter(Z_all[~is_anomaly, 0], Z_all[~is_anomaly, 1], s=16, alpha=0.55, label="normal")
ax.scatter(Z_all[is_anomaly, 0], Z_all[is_anomaly, 1], s=80, marker="x", c="red", label="injected")
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.legend()
ax.set_title("PCA view with anomalies")
plt.show()

## Practice

1. Change PCA to 3 components. How does the reconstruction threshold move?
2. Lower the IsolationForest contamination to 0.01. Which injected points remain flagged?
3. Replace one injected anomaly with a mild point like `[1, 1, 1, 1, 1, 1]`. Does either detector flag it?
4. Print the largest reconstruction-error rows and inspect their original features.

In [ ]:
# Your turn
